4. Extend Grover’s algorithm for a 3-qubit oracle and analyze iterations.

In [1]:
!pip install qiskit
!pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 86.3 MB/s eta 0:00:00


In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import math

def create_oracle():
    # Marks the state |101> (q2=1, q1=0, q0=1)
    qc = QuantumCircuit(3)
    qc.x(1)
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x(1)
    return qc.to_gate(label=" Oracle")

def create_diffuser():
    # Amplifies the amplitude of the marked state
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])
    qc.x([0, 1, 2])
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])
    return qc.to_gate(label=" Diffuser")

# Setup simulator
simulator = Aer.get_backend('qasm_simulator')

# Analyze probabilities across 1 to 3 iterations
for iterations in range(1, 4):
    qc = QuantumCircuit(3, 3)

    # 1. Initialization (equal superposition)
    qc.h([0, 1, 2])

    # 2. Apply Grover iterations
    for _ in range(iterations):
        qc.append(create_oracle(), [0, 1, 2])
        qc.append(create_diffuser(), [0, 1, 2])

    # 3. Measurement
    qc.measure([0, 1, 2], [0, 1, 2])

    # Execute circuit
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit, shots=1024).result()
    counts = result.get_counts()

    print(f"Results after {iterations} iteration(s): {counts}")

Results after 1 iteration(s): {'100': 40, '010': 45, '000': 17, '110': 30, '001': 34, '111': 24, '011': 37, '101': 797}
Results after 2 iteration(s): {'001': 7, '100': 5, '010': 9, '000': 9, '011': 8, '111': 11, '110': 5, '101': 970}
Results after 3 iteration(s): {'011': 105, '111': 103, '100': 92, '001': 98, '000': 105, '010': 94, '110': 104, '101': 323}
